# Score Synthetic Applications

The two fabricated examples share the same application values and have different confirmed terms. This notebook reproduces the point total and recorded-Bad probabilities.

In [1]:
from pathlib import Path
import pandas as pd

repo = Path.cwd()
if not (repo / "models").exists():
    repo = repo.parent

import importlib.util
import json

spec = importlib.util.spec_from_file_location("scorecard", repo / "src/score.py")
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)
model = json.loads((repo / "models/scorecard_spec.json").read_text(encoding="utf-8"))
applications = pd.read_csv(repo / "data/synthetic_applications.csv", dtype="string", keep_default_na=False)
applications

annual_inc,dti,revol_util,purpose,inq_last_6mths,term_months
68000,17.44,50%,debt_consolidation,2,36
68000,17.44,50%,debt_consolidation,2,60


In [2]:
scored = scorer.score_applications(applications, model)
scored

term_months,annual_inc_bin,annual_inc_points,annual_inc_review,dti_bin,dti_points,dti_review,revol_util_bin,revol_util_points,revol_util_review,purpose_bin,purpose_points,purpose_review,inq_last_6mths_bin,inq_last_6mths_points,inq_last_6mths_review,score,probability_bad,review_required
36,"[68,000, 78,000)",1.794902,False,"[17.44, 19.65)",-1.271215,False,"[40.60, 64.90)",0.894565,False,Debt consolidation,-2.414385,False,"[2, 3)",-7.006248,False,554.539991,0.179383,False
60,"[68,000, 78,000)",1.794902,False,"[17.44, 19.65)",-1.271215,False,"[40.60, 64.90)",0.894565,False,Debt consolidation,-2.414385,False,"[2, 3)",-7.006248,False,554.539991,0.333193,False


## Check the published example

The same point total maps to different probabilities through the term intercepts.

In [3]:
import numpy as np

expected = pd.read_csv(repo / "research/scorecard_final_20261121/results/scoring_example.csv")
np.testing.assert_allclose(scored.score, expected.score, rtol=0, atol=1e-10)
np.testing.assert_allclose(scored.probability_bad, expected.probability_bad, rtol=0, atol=1e-12)
scored[["term_months", "score", "probability_bad", "review_required"]]

term_months,score,probability_bad,review_required
36,554.539991,0.179383,False
60,554.539991,0.333193,False
